## Read the three Silver tables 

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
crm_cust = spark.table(
    "e2e_project.silver.crm_cust_info"
)

erp_cust = spark.table(
    "e2e_project.silver.erp_cust_az12"
)

erp_loc = spark.table(
    "e2e_project.silver.erp_customer_location"
)

In [0]:
display(crm_cust.limit(10))
display(erp_cust.limit(10))
display(erp_loc.limit(10))

In [0]:
crm_cust.printSchema()
erp_cust.printSchema()
erp_loc.printSchema()

This is important because we need to know which keys connect the three systems.


## Understand the customer identifiers

cst_id
→ numeric CRM customer identifier
→ sales uses this

cst_key
→ customer business number
→ ERP uses this

## Check the joins BEFORE building Gold

Let's first see whether CRM customers have ERP demographic matches.

In [0]:
crm_vs_erp = (
    crm_cust
    .join(
        erp_cust,
        crm_cust["cst_key"] == erp_cust["customer_id"],
        "left"
    )
)

display(crm_vs_erp.limit(20))

Why left join?

Because CRM is going to be our main customer source.

Every CRM customer
+
ERP information when available

With an inner join, unmatched customers disappear.

CRM customer exists

ERP data missing

        ↓

customer remains

ERP fields = NULL

## Find unmatched ERP customers

quality check

In [0]:
unmatched_erp = (
    crm_cust
    .join(
        erp_cust,
        crm_cust["cst_key"] == erp_cust["customer_id"],
        "left_anti"
    )
)

print("CRM customers without ERP demographics:", unmatched_erp.count())

display(unmatched_erp.limit(20))

## Check location matching too

In [0]:
unmatched_locations = (
    crm_cust
    .join(
        erp_loc,
        crm_cust["cst_key"] == erp_loc["customer_number"],
        "left_anti"
    )
)

print(
    "CRM customers without location:",
    unmatched_locations.count()
)

## Join all three customer sources

In [0]:
customer_joined = (
    crm_cust.alias("crm")
    .join(
        erp_cust.alias("erp"),
        F.col("crm.cst_key") == F.col("erp.customer_id"),
        "left"
    )
    .join(
        erp_loc.alias("loc"),
        F.col("crm.cst_key") == F.col("loc.customer_number"),
        "left"
    )
)

## Resolve gender intelligently

When two systems contain the same attribute, Gold needs an explicit precedence rule.

In [0]:
resolved_gender = (
    F.when(
        (F.col("crm.cst_gndr").isNotNull()) &
        (F.col("crm.cst_gndr") != "n/a"),
        F.col("crm.cst_gndr")
    )
    .when(
        (F.col("erp.gender").isNotNull()) &
        (F.col("erp.gender") != "n/a"),
        F.col("erp.gender")
    )
    .otherwise("n/a")
)

## Build the actual customer dimension columns

In [0]:
dim_customer = customer_joined.select(
    F.col("crm.cst_id").alias("customer_id"),
    F.col("crm.cst_key").alias("customer_number"),

    F.col("crm.cst_firstname").alias("first_name"),
    F.col("crm.cst_lastname").alias("last_name"),

    F.coalesce(
        F.col("loc.country"),
        F.lit("n/a")
    ).alias("country"),

    F.col("crm.cst_marital_status").alias("marital_status"),

    resolved_gender.alias("gender"),

    F.col("erp.birth_date").alias("birth_date"),

    F.col("crm.cst_create_date").alias("create_date")
)

Now instead of source-oriented columns:

cst_id

cst_key

cst_gndr

BDATE

CNTRY
...

Gold becomes:

customer_id

customer_number

first_name

last_name

country

marital_status

gender

birth_date

create_date

That's much more suitable for analytics.

##Why do we need a surrogate key?

We're going to add:

customer_key

This is different from:

customer_id

customer_number

Suppose source systems use:

CRM: 11000

ERP: AW00011000

Those are business/source keys.

The warehouse can instead assign:

customer_key = 1

Another customer:

customer_key = 2

Another:

customer_key = 3

So:

Source identifier     Warehouse identifier

11000             →        1

11001             →        2

11002             →        3

The warehouse-generated identifier is called a surrogate key.

## Generate the surrogate key

In [0]:
customer_key_window = Window.orderBy("customer_id")

dim_customer = dim_customer.withColumn(
    "customer_key",
    F.row_number().over(customer_key_window)
)

In [0]:
dim_customer = dim_customer.select(
    "customer_key",
    "customer_id",
    "customer_number",
    "first_name",
    "last_name",
    "country",
    "marital_status",
    "gender",
    "birth_date",
    "create_date"
)

For this static learning project, row_number() is fine.

In a real incremental production warehouse, surrogate-key management requires more careful persistence because we don't want existing customer keys to change after every load.

## Inspect Gold

In [0]:
display(dim_customer.limit(50))

In [0]:
dim_customer.printSchema()

Silver

Multiple source-oriented tables

Gold

One analytical customer dimension

## Validate uniqueness

In [0]:
display(
    dim_customer
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

check the surrogate key

In [0]:
display(
    dim_customer
    .groupBy("customer_key")
    .count()
    .filter(F.col("count") > 1)
)

## Validate row counts

In [0]:
print("CRM customers:", crm_cust.count())
print("Gold customers:", dim_customer.count())

we probably accidentally created a many-to-many join.

That would be a serious issue.

So row-count validation after joins is extremely important.

## Check NULLs

In [0]:
display(
    dim_customer.select(
        [
            F.sum(
                F.col(c).isNull().cast("int")
            ).alias(c)
            for c in dim_customer.columns
        ]
    )
)

## Save the Gold dimension

In [0]:
(
    dim_customer.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "e2e_project.gold.dim_customer"
    )
)

In [0]:
%sql

SELECT *
FROM e2e_project.gold.dim_customer
LIMIT 20;

That is data integration, not just data cleaning.